## NB_hospital_daily_summary (Silver to Gold)
* **Target:** `<catalog>.gold.hospital_daily_summary`, one row per hospital per calendar day
* **Driving table:** `<catalog>.silver.lab_results`, grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `hospitals` on `hospital_id`, `calendar` on the date key
* **Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date` (day-over-day trend)
* **Load type:** FULL rebuild with `CREATE OR REPLACE TABLE ... AS SELECT`

Implementation notes (based on the silver schema):
* The calendar key is `datekey` (INT, yyyyMMdd), not `date_key`. The join uses `CAST(date_format(test_date, 'yyyyMMdd') AS INT)`.
* `lab_results` contains duplicate `lab_result_id` rows from repeated loads. It is deduplicated to the latest row per key before aggregating.
* Rates are rounded to 2 decimals. `test_volume_change_pct` is NULL when the previous day is NULL or 0. The previous day is the hospital's previous day with tests, not necessarily the previous calendar day.

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
print(f"Building {target_table} from {source_schema}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab test volume, outcome rates and day-over-day trend'
AS
WITH lab AS (
  SELECT *
  FROM {source_schema}.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id
                             ORDER BY update_timestamp DESC, landing_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                              AS summary_date,
    COUNT(*)                               AS total_tests,
    COUNT(DISTINCT patient_id)             AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)              AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM lab
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                                     AS hospital_id,
  h.hospital_name,
  w.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(w.total_tests AS INT)                                                     AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                          AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                             AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                             AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                             AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2))    AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2))    AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                            AS prev_day_total_tests,
  CASE
    WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(ROUND((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100, 2) AS DECIMAL(6,2))
  END                                                                            AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN {source_schema}.calendar c
  ON c.datekey = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))